# Notebook 2: Gradient Boosting Regressor Forecasting Pipeline

This notebook uses a tree-based machine learning model for monthly transaction-volume forecasting. It is an XGBoost-style tabular forecasting architecture implemented with scikit-learn's `GradientBoostingRegressor`, which keeps the PoC lightweight and easy to run without extra compiled dependencies.

The model learns from engineered lag, rolling-window, calendar, and trend features. Forecasting is recursive: each predicted month becomes part of the history used to forecast the next month.

## Runtime Configuration

`MONTHS_HORIZON` maps directly to a future Streamlit slider. Keep it between 1 and 12.

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")

DATASET_PATH = "dataset/credit_card_transactions.csv"
TIMESTAMP_COL = "trans_date_trans_time"
MONTHS_HORIZON = 6
VALIDATION_MONTHS = 3

MONTHS_HORIZON = 12
VALIDATION_MONTHS = 3

## 1. Data Ingestion & Preprocessing

This step loads the transaction timestamp column, aggregates event rows into monthly transaction counts, fills missing calendar months, and removes the incomplete final month.

The output is a single monthly target series named `transaction_volume`.

In [ ]:
from src.time_series_common import MonthlyDataConfig, load_monthly_transaction_volume

config = MonthlyDataConfig(csv_path=DATASET_PATH, timestamp_col=TIMESTAMP_COL)
monthly_series, profile = load_monthly_transaction_volume(config)

print(f"Rows processed: {profile.row_count:,}")
print(f"Invalid timestamps: {profile.invalid_timestamp_count:,}")
print(f"Complete monthly observations: {len(monthly_series)}")

display(monthly_series.rename("monthly_transaction_volume").to_frame())

## 2. Feature Engineering / Stationarity Checks

Machine learning models do not understand time order automatically. We make the time pattern visible through features:

- Lag features: previous 1, 2, 3, and 6 months.
- Rolling means and standard deviation: recent volume level and volatility.
- Month and quarter: calendar position.
- Year index: broad trend proxy.
- Month-over-month growth: recent directional movement.

Stationarity tests are less central for tree models than for SARIMAX because the model consumes supervised features rather than fitting an autoregressive statistical process directly.

In [ ]:
from src.ml_monthly_pipeline import create_supervised_features

feature_frame, target = create_supervised_features(monthly_series)

display(feature_frame.head())
display(target.head().rename("target_transaction_volume").to_frame())

## 3. Model Training & Parameter Tuning

The pipeline tunes a compact grid of gradient boosting parameters and evaluates each candidate on the latest validation months.

This approach is robust for tabular forecasting because the model can capture nonlinear relationships between lagged volumes, rolling averages, and calendar effects.

In [ ]:
from src.ml_monthly_pipeline import train_ml_pipeline

ml_model = train_ml_pipeline(
    monthly_series=monthly_series,
    profile=profile,
    validation_months=VALIDATION_MONTHS,
)

print("Best ML parameters:")
print(ml_model["best_params"])

display(ml_model["metrics"])

## 4. Evaluation Metrics

The validation metrics summarize forecast quality on future-like months:

- **MAE:** average absolute transaction-count miss per month.
- **RMSE:** square-root average squared error, with larger misses receiving more penalty.
- **MAPE:** average percentage miss relative to actual transaction volume.

For a production dashboard, these metrics should be displayed beside the forecast so users understand expected error.

In [ ]:
best_ml_metrics = ml_model["metrics"].iloc[0]

print(f"Validation MAE: {best_ml_metrics['MAE']:,.0f} transactions/month")
print(f"Validation RMSE: {best_ml_metrics['RMSE']:,.0f} transactions/month")
print(f"Validation MAPE: {best_ml_metrics['MAPE']:.2f}%")

## 5. Interactive Forecast Inference

`run_forecast(model, months_horizon)` returns a forecast DataFrame and a Plotly chart. This is the exact pattern a Streamlit app can call from a slider-driven workflow.

In [ ]:
from src.ml_monthly_pipeline import run_forecast

forecast_df, forecast_fig = run_forecast(ml_model, MONTHS_HORIZON)

display(forecast_df)
forecast_fig.show()

## 6. Model Export

The exported pickle contains the fitted gradient boosting model, selected parameters, feature columns, validation metrics, and training metadata.

In [ ]:
from src.ml_monthly_pipeline import export_ml_model

EXPORT_MODEL = True

if EXPORT_MODEL:
    artifact_path = export_ml_model(ml_model, "artifacts/ml_monthly_model.pkl")
    print(f"Model artifact written to: {artifact_path}")

## Developer Insights

### Metric Interpretation

For this model, MAE is the most intuitive metric because the business target is a monthly transaction count. If MAE is 4,000, the forecast is typically off by about 4,000 transactions per month. RMSE tells you whether the model occasionally makes much larger misses. MAPE is useful for stakeholder communication because it expresses error as a percentage of actual monthly volume.

An accurate ML forecast should beat simple baselines and should not produce unstable month-to-month swings unless the underlying transaction history supports them. Because recursive forecasts use earlier predictions as later inputs, error can compound across a 12-month horizon. Treat far-horizon predictions as directional unless validation proves otherwise.

### Model Retraining Strategy

Retrain the ML model every time a new complete month is available. Rebuild all lag and rolling features from the updated monthly series, rerun the parameter grid, compare validation metrics against the previous artifact, and export a new model only if performance is stable or improved.

If new data introduces a structural shift, such as a new customer segment, a product launch, or missing transaction feeds, the lag features may stop representing future behavior. In that case, widen validation analysis, inspect actual vs predicted values month by month, and consider adding segmented features once the dataset has enough history.